# Análise do Corpus e Lacuna Pré-Recuperação — Huawei Ascend RAG

**Objetivo:** caracterizar o corpus de 10 manuais técnicos Huawei (Atlas 200 DK / Ascend 310) **antes** de construir embeddings ou chamar um LLM, e simular como perguntas técnicas reais se alinham lexicalmente aos chunks gerados em `data_chunking.ipynb`.

**Seções do notebook:**
- **Setup:** dependências, caminhos (Google Drive ou local), provisionamento dos dados e tokenizador.
- **Seção 1 — Composição Estrutural:** quanto do corpus é prosa, tabela, código/CLI ou sumário.
- **Seção 2 — Simulação de Mapeamento Pergunta-Chunk:** quantifica a *lacuna pré-recuperação* com e sem breadcrumbs hierárquicos.

**Glossário rápido:**

| Termo | Significado neste notebook |
|---|---|
| **Composição estrutural** | Distribuição dos tokens do documento entre tipos de bloco: prosa narrativa, tabelas, código/CLI e sumário (ToC). |
| **Densidade tabular** | Percentual de tokens BPE que pertencem a tabelas de especificação (excluindo tabelas de sumário). |
| **Lacuna pré-recuperação (pre-retrieval gap)** | Distância entre o vocabulário da pergunta do usuário e o vocabulário do chunk que contém a resposta. Quanto maior a lacuna, maior a chance do recuperador não encontrar o chunk certo. |
| **Sobreposição léxica** | Quantidade de termos compartilhados entre pergunta e chunk (medida aqui por Jaccard, cobertura de termos e BM25). |
| **Breadcrumb** | Caminho hierárquico prefixado ao chunk, por exemplo `[Doc: ATCToolInstructions > Section: 2 Restrictions and Parameters]`. |

> **Importante:** este notebook não usa modelos de embedding nem banco vetorial. Todas as métricas são léxicas e servem como linha de base para os experimentos de pré-consulta (step-back, multi-query / RAG-Fusion, RAG com memória).

**Figuras e artefatos gerados em `data/output/`:**
- `structural_composition_analysis.png`
- `query_mapping_gap_analysis.png`
- `pre_retrieval_analysis_summary.json`

---
# Setup — Ambiente, Caminhos e Dados

As dependências são leves (sem GPU): `tiktoken` para contagem de tokens BPE (`cl100k_base`, o mesmo usado no notebook de chunking), `pandas`/`pyarrow` para leitura do dataset de chunks e `matplotlib` para as figuras.

In [ ]:
# ============================================================================
# Cell S.1 — Dependency Installation
# ============================================================================
%pip install -q tiktoken pandas pyarrow matplotlib

In [ ]:
# ============================================================================
# Cell S.2 — Core Imports & Plot Style
# ============================================================================
import os
import re
import json
import math
import shutil
import urllib.request
from pathlib import Path
from datetime import datetime, timezone
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
import tiktoken
import matplotlib
import matplotlib.pyplot as plt
from IPython.display import display

pd.set_option("display.max_colwidth", 80)
pd.set_option("display.width", 200)

plt.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 200,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.titlelocation": "left",
    "axes.titlesize": 11,
    "axes.labelsize": 9,
    "xtick.labelsize": 8,
    "ytick.labelsize": 8,
    "legend.fontsize": 8,
    "legend.frameon": False,
})
print("[OK] Core imports loaded")

### Persistência e Google Drive

Mesma lógica do `data_chunking.ipynb`:
- `USE_GOOGLE_DRIVE = True` monta o Drive e usa `MyDrive/RAG_Atlas200DK/` como raiz. Todas as figuras e JSONs são gravados diretamente no Drive.
- Fora do Colab, o notebook detecta a ausência de `google.colab` e usa o diretório local do repositório.

In [ ]:
# ============================================================================
# Cell S.3 — Persistence & Directory Configuration
# ============================================================================
USE_GOOGLE_DRIVE = True  # Set to True to read inputs from Drive and save outputs there

if USE_GOOGLE_DRIVE:
    try:
        from google.colab import drive
        try:
            drive.mount("/content/drive")
        except ValueError:
            print("[INFO] Drive already mounted, continuing...")
        except Exception as e:
            print(f"[INFO] Drive mount status: {e}")
    except ImportError:
        print("[INFO] Not running in Google Colab - using local repository paths")
        USE_GOOGLE_DRIVE = False

BASE_DIR = Path("/content/drive/MyDrive/RAG_Atlas200DK") if USE_GOOGLE_DRIVE else Path(".")
PROCESSED_MD_DIR = BASE_DIR / "data" / "processed" / "markdown"
OUTPUT_DIR       = BASE_DIR / "data" / "output"

for d in [PROCESSED_MD_DIR, OUTPUT_DIR]:
    d.mkdir(parents=True, exist_ok=True)

COMPOSITION_PNG = OUTPUT_DIR / "structural_composition_analysis.png"
GAP_PNG         = OUTPUT_DIR / "query_mapping_gap_analysis.png"
SUMMARY_JSON    = OUTPUT_DIR / "pre_retrieval_analysis_summary.json"

print(f"[INFO] BASE_DIR:         {BASE_DIR.resolve()}")
print(f"[INFO] PROCESSED_MD_DIR: {PROCESSED_MD_DIR}")
print(f"[INFO] OUTPUT_DIR:       {OUTPUT_DIR}")

### Provisionamento dos Dados

A célula abaixo garante que os **10 manuais Markdown** e o **dataset de chunks atômicos** (`huawei_ascend_rag_chunks_atomic.parquet` ou `.json`) estejam disponíveis. A ordem de busca é:
1. Diretórios do projeto (`BASE_DIR`), do repositório local e do `/content` no Colab;
2. Download a partir do repositório público no GitHub (os arquivos estão versionados).

Se você regenerou os chunks no `data_chunking.ipynb`, a versão local tem prioridade.

In [ ]:
# ============================================================================
# Cell S.4 — Corpus Asset Provisioning (Markdown manuals + chunk dataset)
# ============================================================================
GITHUB_RAW_BASE = "https://raw.githubusercontent.com/ramonvfs/RAG_Atlas200DK/master/"
CHUNK_DATASET_STEM = "huawei_ascend_rag_chunks_atomic"

EXPECTED_DOCS = [
    "ApplicationDevelopmentGuide_MindStudio_.md",
    "ATCToolInstructions.md",
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_.md",
    "EnvironmentDeploymentGuide.md",
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08.md",
    "MindStudioInstallationGuideUbuntuX86.md",
    "TBECustomOperatorDevelopmentGuide_CLI_.md",
    "TensorFlowNetworkModelPortingandTrainingGuide.md",
    "UserGuide2020.md",
    "UserGuide2021.md",
]

ALT_ROOTS = [
    Path("."),
    Path("/content/RAG_Atlas200DK"),
    Path("/content"),
    Path("/content/drive/MyDrive/RAG_Atlas200DK"),
]


def _download(rel_path: str, dest: Path) -> bool:
    """Download a repository file from GitHub raw into dest. Returns True on success."""
    url = GITHUB_RAW_BASE + urllib.request.quote(rel_path)
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "Mozilla/5.0"})
        with urllib.request.urlopen(req, timeout=60) as resp, open(dest, "wb") as out_f:
            out_f.write(resp.read())
        print(f"  [DOWNLOADED] {rel_path}")
        return True
    except Exception as e:
        print(f"  [WARNING] Could not download {rel_path}: {e}")
        return False


def ensure_corpus_assets(md_dir: Path, out_dir: Path) -> tuple:
    """
    Ensure the 10 processed Markdown manuals and the atomic chunk dataset exist.

    Returns:
        (list of Markdown paths, path of the chunk dataset file)
    """
    # 1. Markdown manuals
    for name in EXPECTED_DOCS:
        target = md_dir / name
        if target.exists():
            continue
        for root in ALT_ROOTS:
            cand = root / "data" / "processed" / "markdown" / name
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {name} from {cand.parent}")
                break
        if not target.exists():
            _download(f"data/processed/markdown/{name}", target)

    md_paths = sorted(p for p in md_dir.glob("*.md") if p.is_file())

    # 2. Chunk dataset (Parquet preferred, JSON fallback)
    dataset_path = None
    for ext in [".parquet", ".json"]:
        target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
        if target.exists():
            dataset_path = target
            break
        for root in ALT_ROOTS:
            cand = root / "data" / "output" / f"{CHUNK_DATASET_STEM}{ext}"
            if cand.exists() and cand.resolve() != target.resolve():
                shutil.copy2(cand, target)
                print(f"  [SYNCED] {target.name} from {cand.parent}")
                dataset_path = target
                break
        if dataset_path:
            break
    if dataset_path is None:
        for ext in [".parquet", ".json"]:
            target = out_dir / f"{CHUNK_DATASET_STEM}{ext}"
            if _download(f"data/output/{CHUNK_DATASET_STEM}{ext}", target):
                dataset_path = target
                break

    return md_paths, dataset_path


def load_chunk_dataset(path: Path) -> list:
    """Load the chunk dataset as a list of dicts (Parquet first, JSON fallback)."""
    if path.suffix == ".parquet":
        try:
            return pd.read_parquet(path).to_dict(orient="records")
        except Exception as e:
            print(f"[WARNING] Parquet read failed ({e}); falling back to JSON")
            json_path = path.with_suffix(".json")
            if not json_path.exists():
                _download(f"data/output/{CHUNK_DATASET_STEM}.json", json_path)
            path = json_path
    with open(path, encoding="utf-8") as f:
        return json.load(f)


md_paths, chunk_dataset_path = ensure_corpus_assets(PROCESSED_MD_DIR, OUTPUT_DIR)
assert len(md_paths) >= 10, f"Expected 10 Markdown manuals, found {len(md_paths)}"
assert chunk_dataset_path is not None, "Chunk dataset not found - run data_chunking.ipynb first"

chunks = load_chunk_dataset(chunk_dataset_path)
chunks_df = pd.DataFrame(chunks)

print(f"\n[OK] Markdown manuals: {len(md_paths)}")
for p in md_paths:
    print(f"   - {p.name} ({p.stat().st_size / 1024:.0f} KB)")
print(f"[OK] Chunk dataset: {chunk_dataset_path.name} -> {len(chunks):,} chunks, "
      f"{chunks_df['doc_name'].nunique()} documents")
print(f"     Columns: {list(chunks_df.columns)}")

In [ ]:
# ============================================================================
# Cell S.5 — Tokenizer & Markdown Normalization Utilities
# ============================================================================
TIKTOKEN_ENCODING = "cl100k_base"
enc = tiktoken.get_encoding(TIKTOKEN_ENCODING)


def count_tokens(text: str) -> int:
    """Count BPE tokens with tiktoken (same encoding as data_chunking.ipynb)."""
    return len(enc.encode(text, disallowed_special=()))


_HTML_TAG_RE = re.compile(r"</?(?:span|sup|sub|br|b|i|u|p|div)\b[^>]*>", re.I)
_MD_ESCAPE_RE = re.compile(r"\\([_*$#\[\](){}<>`|~^.!+\-])")


def normalize_markdown_text(text: str) -> str:
    """
    Normalize marker-generated Markdown for lexical analysis.

    Removes inline HTML anchors (<span id="page-7-0"></span>), unescapes
    Markdown escapes (input\_fp16\_nodes -> input_fp16_nodes) and strips
    bold markers.
    """
    text = _HTML_TAG_RE.sub(" ", text)
    text = _MD_ESCAPE_RE.sub(r"\1", text)
    text = text.replace("**", "")
    return text


SHORT_DOC_LABELS = {
    "ATCToolInstructions": "ATC Tool Instructions",
    "ApplicationDevelopmentGuide_MindStudio_": "App Dev Guide (MindStudio)",
    "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_": "Atlas 200 App SW Dev Guide",
    "EnvironmentDeploymentGuide": "Environment Deployment Guide",
    "Huawei_Atlas_200_DK_developer_kit_Technical_White_Paper__Model_3000__08": "Atlas 200 DK White Paper",
    "MindStudioInstallationGuideUbuntuX86": "MindStudio Install (x86)",
    "TBECustomOperatorDevelopmentGuide_CLI_": "TBE Custom Operator (CLI)",
    "TensorFlowNetworkModelPortingandTrainingGuide": "TF Porting & Training Guide",
    "UserGuide2020": "User Guide 2020",
    "UserGuide2021": "User Guide 2021",
}

_sample = r"Set **input\_fp16\_nodes** <span id=\"page-7-0\"></span>for DaVinci"
print(f"[OK] Tokenizer ready: {TIKTOKEN_ENCODING} ({count_tokens('Ascend 310 AI Processor')} tokens in test)")
print(f"[OK] Normalization test: {normalize_markdown_text(_sample)!r}")

---
# Seção 1 — Análise de Composição Estrutural (Tabelas vs. Código vs. Prosa)

**Por que isso importa para o RAG?** A estratégia de chunking ideal depende do que existe dentro dos documentos:
- **Tabelas** (parâmetros do ATC, pinagem, limites de operadores) perdem o significado quando uma linha é separada do cabeçalho. Um corpus com alta *densidade tabular* justifica o chunking atômico com preservação de tabelas.
- **Código/CLI** (comandos `atc`, configurações AIPP, trechos C++/Python do Matrix e do TE) precisa permanecer íntegro para ser útil como contexto.
- **Prosa** é o caso "fácil": splitters recursivos ou por cabeçalho funcionam bem.

### Como os blocos são classificados

O PDF foi extraído em modo texto (`disable_ocr`), e **nenhum bloco de código saiu cercado por ```` ``` ````**: comandos e trechos de C++/Python aparecem como parágrafos comuns. Por isso o classificador usa heurísticas por linha:

| Categoria | Regra de detecção |
|---|---|
| **Tabela (especificação)** | Linhas consecutivas iniciadas por `|`. |
| **Sumário (ToC)** | Tabela em que pelo menos 50% das linhas têm pontilhado (`.....`) ou o padrão "seção ... página". É mostrada separadamente para não inflar a densidade tabular. |
| **Código / CLI** | Blocos cercados por ```` ``` ```` (se existirem) ou linhas com comandos de shell (`atc`, `sudo`, `pip3`, `export`...), sintaxe de código (`#include`, `import`, `def`, `;` final, `{`/`}`, `//`, `->`, `::`) ou pares de configuração AIPP (`matrix_r0c0 : 298`). |
| **Prosa** | Todo o restante, incluindo títulos. |

Linhas em branco não quebram blocos: um bloco termina quando aparece uma linha não vazia de outra categoria.

In [ ]:
# ============================================================================
# Cell 1.1 — Block Categorization Engine
# ============================================================================
CATEGORIES = ["prose", "table", "code", "toc"]

_FENCE_RE = re.compile(r"^\s*(```|~~~)")
_TOC_DOTS_RE = re.compile(r"\.{5,}")
_TOC_ROW_RE = re.compile(r"^\|\s*(?:\d+(?:\.\d+)*|[A-Z])\s+[^|]+\|\s*\d+\s*\|\s*$")
_SHELL_RE = re.compile(
    r"^\s*(?:\$\s+)?(?:sudo|apt(?:-get)?|pip\d?(?:\.\d+)*|cd|export|chmod|chown|mkdir|cp|mv|rm|tar|"
    r"make|cmake|bash|source|atc|omg|python\d?(?:\.\d+)*|wget|curl|git|dpkg|ln|unzip|su|ssh|scp|"
    r"vi|vim|echo|ls|cat|tail|ifconfig|systemctl|service|mount|umount|fdisk|dd|npu-smi)\s+\S"
)
_CODE_START_RE = re.compile(
    r"^\s*(?:#include\b|#define\b|#if(?:n?def)?\b|import\s+[\w.]+|from\s+[\w.]+\s+import\b|"
    r"def\s+\w+\s*\(|class\s+\w+\s*[:({]|message\s+\w+\s*\{|//|/\*|"
    r"(?:void|int|bool|auto|const|static|struct|enum|typedef|template|namespace|uint\d+_t|"
    r"int\d+_t|HIAI_\w+|Status|std::\w+)\b[^.]*[;{)]\s*$)"
)
_CODE_ANYWHERE_RE = re.compile(r"\w->\w|\w::\w")
_SEMICOLON_END_RE = re.compile(r"[;]\s*$")
_BRACE_ONLY_RE = re.compile(r"^\s*[{}]+\s*;?\s*$")
_CONFIG_KV_RE = re.compile(r"^\s*[a-z][a-z0-9_]*\s*:\s*[-\w.\"/]+\s*,?\s*$")
_INDENTED_RE = re.compile(r"^(?: {4,}|\t)(?![-*+]\s|\d+\.\s)\S")


def is_toc_table(table_lines: list) -> bool:
    """A table is a Table of Contents if >= 50% of its rows look like ToC entries."""
    rows = [l for l in table_lines if l.strip()]
    if not rows:
        return False
    toc_rows = sum(1 for l in rows if _TOC_DOTS_RE.search(l) or _TOC_ROW_RE.match(l.strip()))
    return toc_rows / len(rows) >= 0.5


def classify_line(line: str) -> str:
    """Classify a single non-fenced, non-empty line as 'table', 'code' or 'prose'."""
    if line.lstrip().startswith("|"):
        return "table"
    norm = normalize_markdown_text(line)
    if (_SHELL_RE.match(norm) or _CODE_START_RE.match(norm) or _BRACE_ONLY_RE.match(norm)
            or _CONFIG_KV_RE.match(norm) or _INDENTED_RE.match(line)):
        return "code"
    if _SEMICOLON_END_RE.search(norm) and re.search(r"[(=]|->", norm):
        return "code"
    if _CODE_ANYWHERE_RE.search(norm):
        return "code"
    return "prose"


def categorize_document_blocks(md_text: str, n_samples: int = 3) -> dict:
    """
    Parse a Markdown document into structural blocks and aggregate statistics.

    Block types: prose (narrative text and headings), table (specification
    tables), code (fenced or heuristic code / CLI lines), toc (Table of
    Contents tables). Blank lines do not break blocks.

    Returns:
        dict with one entry per category -> {chars, tokens, blocks, pct_tokens},
        plus 'total_tokens' and 'samples' (first n_samples blocks per category).
    """
    blocks = []                      # list of (category, [lines])
    current_cat, current_lines = None, []
    in_fence = False

    def flush():
        nonlocal current_cat, current_lines
        if current_cat and any(l.strip() for l in current_lines):
            cat = current_cat
            if cat == "table" and is_toc_table(current_lines):
                cat = "toc"
            blocks.append((cat, current_lines))
        current_cat, current_lines = None, []

    for line in md_text.split("\n"):
        if _FENCE_RE.match(line):
            if in_fence:
                current_lines.append(line)
                in_fence = False
                continue
            flush()
            in_fence = True
            current_cat, current_lines = "code", [line]
            continue
        if in_fence:
            current_lines.append(line)
            continue
        if not line.strip():
            if current_cat:
                current_lines.append(line)
            continue
        cat = classify_line(line)
        if cat != current_cat:
            flush()
            current_cat = cat
        current_lines.append(line)
    flush()

    stats = {c: {"chars": 0, "tokens": 0, "blocks": 0, "pct_tokens": 0.0} for c in CATEGORIES}
    samples = {c: [] for c in CATEGORIES}
    for cat, lines in blocks:
        text = "\n".join(lines).strip()
        stats[cat]["chars"] += len(text)
        stats[cat]["tokens"] += count_tokens(text)
        stats[cat]["blocks"] += 1
        if len(samples[cat]) < n_samples:
            samples[cat].append(text[:400])

    total = sum(stats[c]["tokens"] for c in CATEGORIES)
    for c in CATEGORIES:
        stats[c]["pct_tokens"] = round(100.0 * stats[c]["tokens"] / total, 2) if total else 0.0

    return {**stats, "total_tokens": total, "samples": samples}


# --- Unit test on a synthetic snippet ---------------------------------------
_synthetic = "\n".join([
    "# 2 Restrictions and Parameters",
    "The ATC tool converts Caffe and TensorFlow models into offline models.",
    "",
    "| Parameter | Description |",
    "|---|---|",
    "| --input\\_fp16\\_nodes | Input nodes using FP16 |",
    "",
    "atc --model=resnet50.prototxt --framework=0 --soc\\_version=Ascend310",
    "aipp\\_mode : static",
    "matrix\\_r0c0 : 298",
    "",
    "| 1.1 Overview........................ | 3 |",
    "| 1.2 Restrictions.................... | 5 |",
    "",
    "```",
    "import tvm",
    "```",
    "Each cube unit performs a 16x16x16 matrix multiplication.",
])
_res = categorize_document_blocks(_synthetic)
assert _res["table"]["blocks"] == 1, _res["table"]
assert _res["toc"]["blocks"] == 1, _res["toc"]
assert _res["code"]["blocks"] == 2, _res["code"]      # CLI+AIPP block and fenced block
assert _res["prose"]["blocks"] == 2, _res["prose"]
assert abs(sum(_res[c]["pct_tokens"] for c in CATEGORIES) - 100) < 0.1
print("[OK] categorize_document_blocks unit test passed")
print("     " + " | ".join(f"{c}: {_res[c]['blocks']} blocks, {_res[c]['tokens']} tok" for c in CATEGORIES))

In [ ]:
# ============================================================================
# Cell 1.2 — Corpus-Wide Composition Summary
# ============================================================================
composition_rows = []
composition_samples = {}

for md_path in md_paths:
    doc_name = md_path.stem
    result = categorize_document_blocks(md_path.read_text(encoding="utf-8"))
    composition_samples[doc_name] = result["samples"]
    row = {"doc_name": doc_name, "label": SHORT_DOC_LABELS.get(doc_name, doc_name[:30]),
           "total_tokens": result["total_tokens"]}
    for c in CATEGORIES:
        row[f"{c}_tokens"] = result[c]["tokens"]
        row[f"{c}_blocks"] = result[c]["blocks"]
        row[f"{c}_pct"] = result[c]["pct_tokens"]
    composition_rows.append(row)

composition_df = pd.DataFrame(composition_rows)

# Corpus-wide (token-weighted) row
corpus_total = int(composition_df["total_tokens"].sum())
corpus_row = {"doc_name": "CORPUS_TOTAL", "label": "Corpus (token-weighted)", "total_tokens": corpus_total}
for c in CATEGORIES:
    corpus_row[f"{c}_tokens"] = int(composition_df[f"{c}_tokens"].sum())
    corpus_row[f"{c}_blocks"] = int(composition_df[f"{c}_blocks"].sum())
    corpus_row[f"{c}_pct"] = round(100.0 * corpus_row[f"{c}_tokens"] / corpus_total, 2)

composition_df = composition_df.sort_values("table_pct", ascending=False).reset_index(drop=True)
for _, r in composition_df.iterrows():
    assert abs(sum(r[f"{c}_pct"] for c in CATEGORIES) - 100) < 0.1, r["doc_name"]

display_cols = ["label", "total_tokens"] + [f"{c}_pct" for c in CATEGORIES] + [f"{c}_blocks" for c in CATEGORIES]
summary_view = pd.concat([composition_df, pd.DataFrame([corpus_row])], ignore_index=True)[display_cols]

print(f"[OK] Processed {len(composition_df)} manuals | {corpus_total:,} BPE tokens in total\n")
print("Token share per structural category (%), sorted by specification-table density:")
display(summary_view.style.format({c: "{:.1f}" for c in display_cols if c.endswith("_pct")} |
                                  {"total_tokens": "{:,}"}).hide(axis="index"))

print("\nCorpus-wide token share:")
for c in CATEGORIES:
    print(f"   {c:<6} {corpus_row[f'{c}_pct']:6.2f}%  ({corpus_row[f'{c}_tokens']:,} tokens, "
          f"{corpus_row[f'{c}_blocks']:,} blocks)")

In [ ]:
# ============================================================================
# Cell 1.3 — Classifier Sanity Check: Block Samples per Category
# ============================================================================
SAMPLE_DOC = "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide_Models3000_"
_doc_samples = composition_samples.get(SAMPLE_DOC) or next(iter(composition_samples.values()))

for cat in ["table", "code", "toc"]:
    print("=" * 90)
    print(f"Category: {cat.upper()}  (document: {SHORT_DOC_LABELS.get(SAMPLE_DOC, SAMPLE_DOC)})")
    print("=" * 90)
    for i, s in enumerate(_doc_samples[cat][:2], 1):
        print(f"--- sample {i} ---")
        print(s[:300])
    print()

In [ ]:
# ============================================================================
# Cell 1.4 — Visual Dashboard: Structural Composition per Document
# ============================================================================
CATEGORY_STYLE = {
    "prose": {"color": "#5B7DB1", "label": "Narrative prose", "hatch": None},
    "table": {"color": "#2E5E4E", "label": "Specification tables", "hatch": None},
    "code":  {"color": "#C98A2B", "label": "Code / CLI", "hatch": None},
    "toc":   {"color": "#CFCFCF", "label": "Table of Contents (navigation)", "hatch": "///"},
}


def plot_structural_composition(df: pd.DataFrame, corpus: dict, out_path: Path) -> None:
    """Stacked 100% horizontal bars (token share per category) per document plus corpus total."""
    plot_df = pd.concat([pd.DataFrame([corpus]), df], ignore_index=True).iloc[::-1].reset_index(drop=True)
    y = np.arange(len(plot_df))
    fig, ax = plt.subplots(figsize=(11, 6.2))

    left = np.zeros(len(plot_df))
    for c in CATEGORIES:
        vals = plot_df[f"{c}_pct"].to_numpy(dtype=float)
        st = CATEGORY_STYLE[c]
        ax.barh(y, vals, left=left, color=st["color"], hatch=st["hatch"], edgecolor="white",
                linewidth=0.6, label=st["label"], height=0.72)
        for yi, (l, v) in enumerate(zip(left, vals)):
            if v >= 6:
                txt_color = "black" if c == "toc" else "white"
                ax.text(l + v / 2, yi, f"{v:.0f}%", ha="center", va="center", fontsize=7.5, color=txt_color)
        left += vals

    is_corpus = (plot_df["doc_name"] == "CORPUS_TOTAL").to_numpy()
    for yi, (tokens, bold) in enumerate(zip(plot_df["total_tokens"], is_corpus)):
        ax.text(101, yi, f"{tokens / 1000:,.0f}k tok", va="center", fontsize=7.5,
                fontweight="bold" if bold else "normal", color="#444")
    ax.set_yticks(y)
    ax.set_yticklabels(plot_df["label"])
    for tick, bold in zip(ax.get_yticklabels(), is_corpus):
        if bold:
            tick.set_fontweight("bold")
    ax.axhline(len(plot_df) - 1.5, color="#888", linewidth=0.8, linestyle="--")
    ax.set_xlim(0, 100)
    ax.set_xlabel("Share of BPE tokens (cl100k_base), %")
    ax.set_title("Structural composition of the Huawei Ascend corpus (10 manuals, sorted by "
                 "specification-table density)")
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.09), ncol=4)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"[SAVE] {out_path}")


plot_structural_composition(composition_df, corpus_row, COMPOSITION_PNG)

### Como interpretar a composição estrutural

- **Densidade tabular alta** (por exemplo, guias de API e do ATC) indica que boa parte das respostas está em linhas de tabela. Nesses documentos, cortar uma tabela no meio separa o valor do seu cabeçalho, e o chunk passa a ser ambíguo. Isso justifica a estratégia `3_atomic_table_aware`, que mantém tabelas inteiras.
- **Código/CLI** é minoritário em tokens, mas concentra perguntas de altíssimo valor (flags exatas do `atc`, parâmetros AIPP, chamadas `HIAI_*`). São justamente os termos que a busca léxica (BM25) encontra bem e que embeddings densos tendem a diluir, um argumento para busca híbrida nas próximas etapas.
- **Sumários (ToC)** não contêm respostas, mas compartilham muitos termos com as perguntas (títulos de seção). Eles competem com os chunks de conteúdo na recuperação e são candidatos a filtragem por metadado.
- **Prosa** domina documentos de instalação e guias de usuário, onde splitters por cabeçalho já preservam bem o contexto.

---
# Seção 2 — Simulação de Mapeamento Pergunta-Chunk e Análise da Lacuna Pré-Recuperação

**Pergunta central:** antes de qualquer embedding, o vocabulário das perguntas técnicas encontra o chunk que contém a resposta? E quanto o **breadcrumb hierárquico** ajuda?

### Desenho do experimento

1. **Benchmark de 25 perguntas** em 5 arquétipos de domínio (5 perguntas cada):
   - `exact_parameter_flags` — flags exatas do ATC (`--input_fp16_nodes`, `--precision_mode`).
   - `hardware_constraints` — revisões de placa, alimentação, TOPS, memória.
   - `aipp_preprocessing` — CSC, crop, padding, normalização.
   - `troubleshooting` — mensagens de erro e falhas de conexão/conversão.
   - `tensor_layout_architecture` — NC1HWC0, FRACTAL_Z, Cube Unit.

   Perguntas marcadas com `scope=True` são **genéricas ou dependentes de escopo** (não citam o termo exato), simulando o usuário que não conhece o vocabulário do manual.
2. **Gabarito (ground truth):** cada pergunta tem um filtro de documento (`gold_docs`) e uma expressão regular (`gold_pattern`) aplicada **somente ao corpo do chunk, nunca ao breadcrumb**, para não favorecer o Modo B. Chunks de sumário são excluídos do gabarito.
3. **Dois modos de indexação:**
   - **Modo A — Conteúdo bruto:** apenas o corpo do chunk (a linha de breadcrumb embutida é removida).
   - **Modo B — Enriquecido com breadcrumb:** `[Doc: ... > Section: ...]` + corpo.
4. **Pontuadores léxicos:** BM25 (k1=1.5, b=0.75) como ranqueador principal e Jaccard como ranqueador secundário.

### Métricas

| Métrica | Definição |
|---|---|
| **Hit@1 / Hit@3** | Fração de perguntas em que algum chunk do gabarito aparece na 1a posição / entre as 3 primeiras. |
| **MRR@10** | Média de 1/posição do primeiro chunk correto (0 se fora do top-10). |
| **Cobertura de termos** | Fração dos termos da pergunta presentes no melhor chunk do gabarito. |
| **Jaccard** | Interseção / união entre os conjuntos de termos da pergunta e do chunk. |
| **Breadcrumb lift** | Ganho absoluto (pontos percentuais) e relativo do Modo B sobre o Modo A. |

> **Limitação:** quando o gabarito repete um termo da pergunta (por exemplo, `CheckPartitionSpace`), a pergunta é lexicalmente "fácil" por construção. Isso é intencional para flags e mensagens de erro, que é exatamente onde a busca léxica brilha. As perguntas `scope=True` medem o caso difícil. Os números aqui são uma linha de base léxica, **não** um resultado de recuperação densa.

In [ ]:
# ============================================================================
# Cell 2.1 — Benchmark Query Suite (25 queries, 5 archetypes)
# ============================================================================
ATC   = "ATCToolInstructions"
ATLAS = "Atlas200AIAcceleratorModuleApplicationSoftwareDevelopmentGuide"
WHITE = "White_Paper"
ENV   = "EnvironmentDeploymentGuide"
TF    = "TensorFlowNetworkModelPortingandTrainingGuide"
TBE   = "TBECustomOperatorDevelopmentGuide"
UG21  = "UserGuide2021"
UG20  = "UserGuide2020"

BENCHMARK_QUERIES = [
    # --- exact_parameter_flags -------------------------------------------------------------
    {"id": "Q01", "archetype": "exact_parameter_flags", "scope": False,
     "query": "What does the --input_fp16_nodes parameter do in the ATC tool?",
     "gold_docs": [ATC], "gold_pattern": r"input_fp16_nodes"},
    {"id": "Q02", "archetype": "exact_parameter_flags", "scope": False,
     "query": "How do I set --precision_mode=force_fp16 when converting a model?",
     "gold_docs": [ATC, TF], "gold_pattern": r"force_fp16"},
    {"id": "Q03", "archetype": "exact_parameter_flags", "scope": False,
     "query": "What does the insert_op_conf parameter point to?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"insert_op_conf"},
    {"id": "Q04", "archetype": "exact_parameter_flags", "scope": False,
     "query": "How do I specify the output nodes of a model with --out_nodes?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"out_nodes"},
    {"id": "Q05", "archetype": "exact_parameter_flags", "scope": True,
     "query": "Which SoC version value must be passed for this board?",
     "gold_docs": [ATC], "gold_pattern": r"(?s)soc_version.*Ascend310|Ascend310.*soc_version"},

    # --- hardware_constraints --------------------------------------------------------------
    {"id": "Q06", "archetype": "hardware_constraints", "scope": False,
     "query": "What are the differences between the IT21DMDA and IT21VDMB board versions?",
     "gold_docs": [WHITE], "gold_pattern": r"IT21VDMB"},
    {"id": "Q07", "archetype": "hardware_constraints", "scope": True,
     "query": "What input voltage range does the power port of the developer kit accept?",
     "gold_docs": [WHITE], "gold_pattern": r"28 ?V"},
    {"id": "Q08", "archetype": "hardware_constraints", "scope": False,
     "query": "What is the INT8 computing power in TOPS of the Ascend 310?",
     "gold_docs": [WHITE], "gold_pattern": r"TOPS"},
    {"id": "Q09", "archetype": "hardware_constraints", "scope": False,
     "query": "How much LPDDR4X memory does the developer kit provide?",
     "gold_docs": [WHITE], "gold_pattern": r"LPDDR4X"},
    {"id": "Q10", "archetype": "hardware_constraints", "scope": True,
     "query": "Which USB port type does the board expose?",
     "gold_docs": [WHITE], "gold_pattern": r"USB"},

    # --- aipp_preprocessing ----------------------------------------------------------------
    {"id": "Q11", "archetype": "aipp_preprocessing", "scope": False,
     "query": "Which CSC matrix coefficients convert YUV420SP to RGB888 in AIPP?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"matrix_r0c0"},
    {"id": "Q12", "archetype": "aipp_preprocessing", "scope": False,
     "query": "How do I enable color space conversion with csc_switch?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"csc_switch"},
    {"id": "Q13", "archetype": "aipp_preprocessing", "scope": False,
     "query": "How are crop start positions set with load_start_pos_w and load_start_pos_h?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"load_start_pos"},
    {"id": "Q14", "archetype": "aipp_preprocessing", "scope": True,
     "query": "How is padding configured in the image preprocessing configuration file?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"padding_size"},
    {"id": "Q15", "archetype": "aipp_preprocessing", "scope": True,
     "query": "How do I normalize the mean and variance of the input image before inference?",
     "gold_docs": [ATC, ATLAS], "gold_pattern": r"var_reci_chn"},

    # --- troubleshooting -------------------------------------------------------------------
    {"id": "Q16", "archetype": "troubleshooting", "scope": False,
     "query": "Driver upgrade fails with CheckPartitionSpace partition space check failed",
     "gold_docs": [ENV], "gold_pattern": r"CheckPartitionSpace"},
    {"id": "Q17", "archetype": "troubleshooting", "scope": False,
     "query": "Model conversion of a frozen graph with dynamic shape input fails",
     "gold_docs": [ATC, ATLAS, TF], "gold_pattern": r"(?is)frozen.*dynamic|dynamic.*frozen"},
    {"id": "Q18", "archetype": "troubleshooting", "scope": False,
     "query": "Why is the destructor not HIAI_DFree when memory is allocated with HIAI_DMalloc?",
     "gold_docs": [ATLAS], "gold_pattern": r"(?i)destructor"},
    {"id": "Q19", "archetype": "troubleshooting", "scope": True,
     "query": "The board cannot connect to the Ubuntu server, what should I do?",
     "gold_docs": [UG21, UG20, ENV], "gold_pattern": r"(?i)cannot connect to the ubuntu server"},
    {"id": "Q20", "archetype": "troubleshooting", "scope": True,
     "query": "What restrictions apply when converting a model?",
     "gold_docs": [ATC], "gold_pattern": r"(?im)^#+.*restrictions"},

    # --- tensor_layout_architecture --------------------------------------------------------
    {"id": "Q21", "archetype": "tensor_layout_architecture", "scope": False,
     "query": "How is the NC1HWC0 5D format derived from NCHW?",
     "gold_docs": [ATC, ATLAS, TBE], "gold_pattern": r"NC1HWC0"},
    {"id": "Q22", "archetype": "tensor_layout_architecture", "scope": True,
     "query": "What is the C0 value for float16 data in the 5D format?",
     "gold_docs": [ATC, ATLAS, TBE], "gold_pattern": r"(?s)C0.*(?:16|float16)"},
    {"id": "Q23", "archetype": "tensor_layout_architecture", "scope": False,
     "query": "What does the Cube Unit of the DaVinci AI Core compute?",
     "gold_docs": [ATC, ATLAS, TBE], "gold_pattern": r"(?i)cube unit"},
    {"id": "Q24", "archetype": "tensor_layout_architecture", "scope": False,
     "query": "What is the FRACTAL_Z format used for?",
     "gold_docs": [ATC, ATLAS, TBE], "gold_pattern": r"(?i)fractal_?z"},
    {"id": "Q25", "archetype": "tensor_layout_architecture", "scope": True,
     "query": "What is the difference between the two 4D layouts NCHW and NHWC?",
     "gold_docs": [ATC, ATLAS, TBE], "gold_pattern": r"NHWC"},
]

ARCHETYPES = ["exact_parameter_flags", "hardware_constraints", "aipp_preprocessing",
              "troubleshooting", "tensor_layout_architecture"]
ARCHETYPE_LABELS = {
    "exact_parameter_flags": "Exact flags\n& params",
    "hardware_constraints": "Hardware\nconstraints",
    "aipp_preprocessing": "AIPP\npreproc.",
    "troubleshooting": "Troubleshoot\n& errors",
    "tensor_layout_architecture": "Tensor layout\n& arch.",
}

assert len(BENCHMARK_QUERIES) == 25
assert all(sum(q["archetype"] == a for q in BENCHMARK_QUERIES) == 5 for a in ARCHETYPES)
print(f"[OK] Benchmark suite: {len(BENCHMARK_QUERIES)} queries | "
      f"{sum(q['scope'] for q in BENCHMARK_QUERIES)} scope-dependent (generic) queries")
display(pd.DataFrame(BENCHMARK_QUERIES)[["id", "archetype", "scope", "query"]].style.hide(axis="index"))

In [ ]:
# ============================================================================
# Cell 2.2 — Chunk Text Views & Ground-Truth Resolution
# ============================================================================
MIN_GOLD, MAX_GOLD = 1, 30


def split_chunk_body(content: str) -> str:
    """Remove the breadcrumb line that data_chunking.ipynb embeds at the top of each chunk."""
    first, _, rest = content.partition("\n")
    if first.startswith("[Doc:"):
        return rest.strip()
    return content.strip()


def is_toc_chunk(body: str) -> bool:
    """True if the chunk body is predominantly a Table of Contents."""
    table_lines = [l for l in body.split("\n") if l.lstrip().startswith("|")]
    return len(table_lines) >= 2 and is_toc_table(table_lines)


for ch in chunks:
    ch["body"] = split_chunk_body(ch["content"])
    ch["body_norm"] = normalize_markdown_text(ch["body"])
    ch["breadcrumb_norm"] = normalize_markdown_text(ch["breadcrumb"])
    ch["is_toc"] = is_toc_chunk(ch["body"])


def resolve_gold_chunks(query: dict, chunk_list: list) -> list:
    """Indices of chunks whose doc matches gold_docs and whose BODY matches gold_pattern."""
    pattern = re.compile(query["gold_pattern"])
    return [i for i, ch in enumerate(chunk_list)
            if not ch["is_toc"]
            and any(g in ch["doc_name"] for g in query["gold_docs"])
            and pattern.search(ch["body_norm"])]


for q in BENCHMARK_QUERIES:
    q["gold_idx"] = resolve_gold_chunks(q, chunks)

gold_view = pd.DataFrame([{
    "id": q["id"], "archetype": q["archetype"], "gold_count": len(q["gold_idx"]),
    "gold_docs": sorted({SHORT_DOC_LABELS.get(chunks[i]["doc_name"], chunks[i]["doc_name"]) for i in q["gold_idx"]}),
} for q in BENCHMARK_QUERIES])
display(gold_view.style.hide(axis="index"))

bad = gold_view[(gold_view["gold_count"] < MIN_GOLD) | (gold_view["gold_count"] > MAX_GOLD)]
assert bad.empty, f"Gold sets out of bounds [{MIN_GOLD}, {MAX_GOLD}]:\n{bad}"
print(f"[OK] All queries resolve to {MIN_GOLD}..{MAX_GOLD} gold chunks "
      f"(median {int(gold_view['gold_count'].median())}) | ToC chunks in corpus: {sum(c['is_toc'] for c in chunks)}")

In [ ]:
# ============================================================================
# Cell 2.3 — Lexical Tokenizer, BM25 Index and Jaccard Similarity
# ============================================================================
STOPWORDS = set("""
a an the of to in on for and or is are was were be been being what which who whom how why when where
does do did can could should would will shall may might must this that these those it its with by from
at as into about than then there their them they i my we our you your if not no so such any all each
use used using via per me he she his her section doc
""".split())

_LEX_TOKEN_RE = re.compile(r"[A-Za-z0-9]+(?:[_\-.][A-Za-z0-9]+)*")
_CAMEL_RE = re.compile(r"[A-Z]+(?=[A-Z][a-z])|[A-Z]?[a-z0-9]+|[A-Z0-9]+")


def lexical_tokenize(text: str) -> list:
    """
    Tokenize technical text for lexical matching.

    Keeps compound identifiers whole (input_fp16_nodes, yuv420sp, 3.0) and also
    emits their sub-tokens split on '_', '-', '.' and camelCase
    (ATCToolInstructions -> atc, tool, instructions), so document names in
    breadcrumbs can match natural-language query words.
    """
    out = []
    for m in _LEX_TOKEN_RE.findall(normalize_markdown_text(text)):
        low = m.lower()
        if low not in STOPWORDS:
            out.append(low)
        parts = [p for p in re.split(r"[_\-.]", m) if p]
        subs = []
        for p in parts:
            subs.extend(_CAMEL_RE.findall(p) if re.search(r"[a-z][A-Z]", p) else [p])
        if len(subs) > 1:
            out.extend(s.lower() for s in subs if s.lower() not in STOPWORDS and len(s) > 1)
    return out


class BM25:
    """Okapi BM25 over pre-tokenized documents with an inverted index (lesson 09 style)."""

    def __init__(self, docs_tokens: list, k1: float = 1.5, b: float = 0.75):
        self.k1, self.b = k1, b
        self.n_docs = len(docs_tokens)
        self.lens = np.array([len(t) for t in docs_tokens], dtype=float)
        self.avg_len = float(self.lens.mean())
        self.postings = defaultdict(list)          # term -> [(doc_idx, tf)]
        for idx, toks in enumerate(docs_tokens):
            for term, tf in Counter(toks).items():
                self.postings[term].append((idx, tf))
        self.idf = {t: math.log(1 + (self.n_docs - len(p) + 0.5) / (len(p) + 0.5))
                    for t, p in self.postings.items()}

    def scores(self, query: str) -> np.ndarray:
        s = np.zeros(self.n_docs)
        for term in set(lexical_tokenize(query)):
            if term not in self.postings:
                continue
            idf = self.idf[term]
            for idx, tf in self.postings[term]:
                norm = self.k1 * (1 - self.b + self.b * self.lens[idx] / self.avg_len)
                s[idx] += idf * tf * (self.k1 + 1) / (tf + norm)
        return s


def jaccard(a: set, b: set) -> float:
    """Jaccard similarity between two token sets."""
    return len(a & b) / len(a | b) if (a or b) else 0.0


def term_coverage(query_terms: set, doc_terms: set) -> float:
    """Fraction of query terms present in the document."""
    return len(query_terms & doc_terms) / len(query_terms) if query_terms else 0.0


_t = lexical_tokenize("What does --input_fp16_nodes do in [Doc: ATCToolInstructions > Section: Parameters]?")
assert "input_fp16_nodes" in _t and "fp16" in _t and "atc" in _t and "instructions" in _t, _t
print(f"[OK] Tokenizer test: {_t}")

In [ ]:
# ============================================================================
# Cell 2.4 — Lexical Alignment Evaluator (Mode A: raw vs Mode B: breadcrumb)
# ============================================================================
MODES = {
    "raw": lambda ch: ch["body"],
    "breadcrumb": lambda ch: ch["breadcrumb"] + "\n" + ch["body"],
}
MRR_CUTOFF = 10


def _first_gold_rank(order: np.ndarray, gold: set, max_rank: int = 200):
    """1-based rank of the first gold chunk in a ranking, or None if beyond max_rank."""
    for rank, idx in enumerate(order[:max_rank], 1):
        if idx in gold:
            return rank
    return None


def evaluate_query_chunk_alignment(queries: list, chunks: list, k1: float = 1.5, b: float = 0.75) -> dict:
    """
    Benchmark lexical alignment between queries and chunks in two indexing modes.

    Mode A ('raw') indexes only the chunk body; Mode B ('breadcrumb') prepends
    the hierarchical breadcrumb. For each mode the function ranks all chunks
    with BM25 (primary) and Jaccard (secondary) and records Hit@1, Hit@3,
    MRR@10, plus Jaccard and term coverage against the best gold chunk.

    Returns:
        dict with 'per_query' (DataFrame), 'per_archetype' (DataFrame),
        'overall' (dict of raw / breadcrumb / lift metrics) and 'config'.
    """
    mode_tokens = {m: [lexical_tokenize(fn(ch)) for ch in chunks] for m, fn in MODES.items()}
    mode_sets = {m: [set(t) for t in toks] for m, toks in mode_tokens.items()}
    mode_bm25 = {m: BM25(toks, k1=k1, b=b) for m, toks in mode_tokens.items()}

    records = []
    for q in queries:
        gold = set(q["gold_idx"])
        q_terms = set(lexical_tokenize(q["query"]))
        rec = {"id": q["id"], "archetype": q["archetype"], "scope": q["scope"],
               "query": q["query"], "gold_count": len(gold), "query_terms": len(q_terms)}
        for m in MODES:
            bm25_order = np.argsort(-mode_bm25[m].scores(q["query"]), kind="stable")
            jac_scores = np.array([jaccard(q_terms, s) for s in mode_sets[m]])
            jac_order = np.argsort(-jac_scores, kind="stable")

            rank = _first_gold_rank(bm25_order, gold)
            jrank = _first_gold_rank(jac_order, gold)
            rec[f"bm25_rank_{m}"] = rank
            rec[f"hit1_{m}"] = int(rank is not None and rank <= 1)
            rec[f"hit3_{m}"] = int(rank is not None and rank <= 3)
            rec[f"rr_{m}"] = 1.0 / rank if rank is not None and rank <= MRR_CUTOFF else 0.0
            rec[f"jac_hit1_{m}"] = int(jrank is not None and jrank <= 1)
            rec[f"jac_hit3_{m}"] = int(jrank is not None and jrank <= 3)
            rec[f"gold_jaccard_{m}"] = max(jaccard(q_terms, mode_sets[m][i]) for i in gold)
            rec[f"gold_coverage_{m}"] = max(term_coverage(q_terms, mode_sets[m][i]) for i in gold)
            top = chunks[int(bm25_order[0])]
            rec[f"top1_{m}"] = top["chunk_id"]
            rec[f"top1_section_{m}"] = top["breadcrumb_norm"][:110]
        records.append(rec)

    per_query = pd.DataFrame(records)

    metric_map = {"hit@1": "hit1", "hit@3": "hit3", "mrr@10": "rr", "jaccard_hit@3": "jac_hit3",
                  "gold_jaccard": "gold_jaccard", "gold_coverage": "gold_coverage"}

    def summarize(df: pd.DataFrame) -> dict:
        out = {m: {name: round(float(df[f"{col}_{m}"].mean()), 4) for name, col in metric_map.items()}
               for m in MODES}
        lift = {}
        for name in metric_map:
            a, b_ = out["raw"][name], out["breadcrumb"][name]
            lift[f"{name}_abs_pp"] = round(100 * (b_ - a), 2)
            lift[f"{name}_rel_pct"] = round(100 * (b_ - a) / a, 2) if a > 0 else None
        out["lift"] = lift
        out["n_queries"] = int(len(df))
        return out

    per_arch_rows, per_arch_dict = [], {}
    for arch in ARCHETYPES:
        s = summarize(per_query[per_query["archetype"] == arch])
        per_arch_dict[arch] = s
        per_arch_rows.append({"archetype": arch,
                              **{f"{k}_raw": v for k, v in s["raw"].items()},
                              **{f"{k}_breadcrumb": v for k, v in s["breadcrumb"].items()},
                              "hit@3_lift_pp": s["lift"]["hit@3_abs_pp"],
                              "hit@1_lift_pp": s["lift"]["hit@1_abs_pp"]})

    return {
        "per_query": per_query,
        "per_archetype": pd.DataFrame(per_arch_rows),
        "per_archetype_dict": per_arch_dict,
        "overall": summarize(per_query),
        "scope_split": {("scope" if flag else "explicit"): summarize(per_query[per_query["scope"] == flag])
                        for flag in [False, True]},
        "config": {"scorer": f"bm25(k1={k1},b={b})", "secondary_scorer": "jaccard",
                   "mrr_cutoff": MRR_CUTOFF, "modes": list(MODES), "n_chunks": len(chunks)},
    }


alignment = evaluate_query_chunk_alignment(BENCHMARK_QUERIES, chunks)
ov = alignment["overall"]
print(f"[OK] Evaluated {ov['n_queries']} queries against {len(chunks):,} chunks\n")
print(f"{'Metric':<16}{'Raw':>10}{'Breadcrumb':>13}{'Lift (pp)':>12}")
for name in ["hit@1", "hit@3", "mrr@10", "jaccard_hit@3", "gold_coverage", "gold_jaccard"]:
    print(f"{name:<16}{ov['raw'][name]:>10.3f}{ov['breadcrumb'][name]:>13.3f}"
          f"{ov['lift'][name + '_abs_pp']:>+12.1f}")

In [ ]:
# ============================================================================
# Cell 2.5 — Per-Query Results
# ============================================================================
pq = alignment["per_query"]
per_query_view = pq[["id", "archetype", "scope", "gold_count", "bm25_rank_raw", "bm25_rank_breadcrumb",
                     "hit3_raw", "hit3_breadcrumb", "gold_coverage_raw", "gold_coverage_breadcrumb",
                     "top1_section_raw"]].copy()
per_query_view["bm25_rank_raw"] = per_query_view["bm25_rank_raw"].map(lambda r: ">200" if pd.isna(r) else int(r))
per_query_view["bm25_rank_breadcrumb"] = per_query_view["bm25_rank_breadcrumb"].map(
    lambda r: ">200" if pd.isna(r) else int(r))
display(per_query_view.style.format({"gold_coverage_raw": "{:.2f}", "gold_coverage_breadcrumb": "{:.2f}"})
        .hide(axis="index"))

### Diagnóstico da lacuna por arquétipo

A célula a seguir ordena os arquétipos pela **taxa de falha no Modo A** (sem breadcrumbs) e separa as perguntas em quatro grupos:
- **Recuperadas apenas com breadcrumb:** o escopo hierárquico (nome do documento + seções pai) fechou a lacuna.
- **Prejudicadas pelo breadcrumb:** o prefixo adicionou termos genéricos que favoreceram chunks errados.
- **Perdidas nos dois modos:** a lacuna é de vocabulário, não de escopo. São as candidatas naturais às técnicas de pré-consulta (step-back, multi-query / RAG-Fusion, reescrita).
- **Encontradas nos dois modos:** perguntas com termos exatos, onde a busca léxica já resolve.

In [ ]:
# ============================================================================
# Cell 2.6 — Results Reporting & Gap Diagnosis by Archetype
# ============================================================================
pa = alignment["per_archetype"].copy()
pa["miss_rate@3_raw"] = 1 - pa["hit@3_raw"]
pa = pa.sort_values(["miss_rate@3_raw", "hit@1_raw"], ascending=[False, True]).reset_index(drop=True)

view_cols = ["archetype", "hit@1_raw", "hit@1_breadcrumb", "hit@3_raw", "hit@3_breadcrumb",
             "mrr@10_raw", "mrr@10_breadcrumb", "gold_coverage_raw", "gold_coverage_breadcrumb",
             "hit@3_lift_pp", "miss_rate@3_raw"]
print("Archetypes ranked by pre-retrieval gap (Mode A miss rate @3):")
display(pa[view_cols].style.format({c: "{:.2f}" for c in view_cols if c != "archetype"}).hide(axis="index"))

pq = alignment["per_query"]
groups = {
    "Recovered only with breadcrumbs": pq[(pq["hit3_breadcrumb"] == 1) & (pq["hit3_raw"] == 0)],
    "Degraded by breadcrumbs": pq[(pq["hit3_breadcrumb"] == 0) & (pq["hit3_raw"] == 1)],
    "Missed in both modes (query-transformation candidates)": pq[(pq["hit3_breadcrumb"] == 0) & (pq["hit3_raw"] == 0)],
    "Found in both modes": pq[(pq["hit3_breadcrumb"] == 1) & (pq["hit3_raw"] == 1)],
}
for title, df in groups.items():
    print("\n" + "=" * 100)
    print(f"{title}: {len(df)} queries")
    print("=" * 100)
    for _, r in df.iterrows():
        rr = "-" if pd.isna(r["bm25_rank_raw"]) else int(r["bm25_rank_raw"])
        rb = "-" if pd.isna(r["bm25_rank_breadcrumb"]) else int(r["bm25_rank_breadcrumb"])
        print(f"  {r['id']} [{r['archetype']:<27}] scope={str(r['scope']):<5} rank raw={rr:<4} "
              f"bc={rb:<4} | {r['query'][:70]}")

worst = pa.iloc[0]
print("\n[DIAGNOSIS] Largest pre-retrieval gap without breadcrumbs: "
      f"{worst['archetype']} (Hit@3 raw = {worst['hit@3_raw']:.2f}, "
      f"breadcrumb = {worst['hit@3_breadcrumb']:.2f}, lift = {worst['hit@3_lift_pp']:+.0f} pp)")
ss = alignment["scope_split"]
print(f"[DIAGNOSIS] Explicit-term queries: Hit@3 raw = {ss['explicit']['raw']['hit@3']:.2f} -> "
      f"breadcrumb = {ss['explicit']['breadcrumb']['hit@3']:.2f}")
print(f"[DIAGNOSIS] Scope-dependent queries: Hit@3 raw = {ss['scope']['raw']['hit@3']:.2f} -> "
      f"breadcrumb = {ss['scope']['breadcrumb']['hit@3']:.2f}")

In [ ]:
# ============================================================================
# Cell 2.7 — Comparison Plot: Raw Content vs Breadcrumb Enriched
# ============================================================================
MODE_STYLE = {
    "raw": {"color": "#9A9A9A", "label": "Raw content (Mode A)"},
    "breadcrumb": {"color": "#2E5E4E", "label": "Breadcrumb enriched (Mode B)"},
}


def plot_gap_analysis(result: dict, out_path: Path) -> None:
    """Grouped bars of Hit@1 and Hit@3 per archetype (plus overall) for both indexing modes."""
    per_arch = result["per_archetype_dict"]
    overall = result["overall"]
    groups = ARCHETYPES + ["overall"]
    labels = [ARCHETYPE_LABELS[a] for a in ARCHETYPES] + ["Overall\n(25 queries)"]
    x = np.arange(len(groups), dtype=float)
    x[-1] += 0.4                                   # visual gap before the overall group
    w = 0.38

    fig, axes = plt.subplots(1, 2, figsize=(13.5, 4.8), sharey=True)
    for ax, metric, title in zip(axes, ["hit@1", "hit@3"], ["Top-1 hit rate", "Top-3 hit rate"]):
        for j, mode in enumerate(["raw", "breadcrumb"]):
            vals = [(overall if g == "overall" else per_arch[g])[mode][metric] for g in groups]
            bars = ax.bar(x + (j - 0.5) * w, vals, w, color=MODE_STYLE[mode]["color"],
                          label=MODE_STYLE[mode]["label"])
            for bar, v in zip(bars, vals):
                ax.text(bar.get_x() + bar.get_width() / 2, v + 0.015, f"{v:.0%}", ha="center",
                        va="bottom", fontsize=7, color="#333")
        for xi, g in zip(x, groups):
            src = overall if g == "overall" else per_arch[g]
            lift = src["lift"][f"{metric}_abs_pp"]
            color = "#2E5E4E" if lift > 0 else ("#A63D40" if lift < 0 else "#777")
            ax.text(xi, 1.13, f"{lift:+.0f} pp", ha="center", fontsize=8, fontweight="bold", color=color)
        ax.axvline(x[-2] + 0.7, color="#BBB", linewidth=0.8, linestyle="--")
        ax.set_xticks(x)
        ax.set_xticklabels(labels, fontsize=8)
        ax.set_ylim(0, 1.22)
        ax.set_yticks(np.linspace(0, 1, 6))
        ax.yaxis.set_major_formatter(matplotlib.ticker.PercentFormatter(1.0))
        ax.set_title(f"{title} (BM25)  |  breadcrumb lift shown above bars")
    axes[0].set_ylabel("Queries with a gold chunk retrieved")
    axes[0].legend(loc="upper center", bbox_to_anchor=(1.05, -0.2), ncol=2)
    fig.suptitle(f"Pre-retrieval gap: lexical retrieval on {result['config']['n_chunks']:,} atomic chunks, "
                 f"raw content vs breadcrumb-enriched", x=0.01, ha="left", fontsize=12)
    plt.tight_layout()
    plt.savefig(out_path, bbox_inches="tight")
    plt.show()
    print(f"[SAVE] {out_path}")


plot_gap_analysis(alignment, GAP_PNG)

In [ ]:
# ============================================================================
# Cell 2.8 — Export Metrics Summary (JSON)
# ============================================================================
def _to_builtin(obj):
    """Recursively convert numpy / pandas scalars into JSON-serializable Python types."""
    if isinstance(obj, dict):
        return {str(k): _to_builtin(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_to_builtin(v) for v in obj]
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return None if np.isnan(obj) else float(obj)
    if isinstance(obj, float) and math.isnan(obj):
        return None
    return obj


per_query_export = []
for q, (_, r) in zip(BENCHMARK_QUERIES, alignment["per_query"].iterrows()):
    per_query_export.append({
        "id": q["id"], "archetype": q["archetype"], "scope": q["scope"], "query": q["query"],
        "gold_docs": q["gold_docs"], "gold_pattern": q["gold_pattern"], "gold_count": len(q["gold_idx"]),
        "bm25_rank_raw": None if pd.isna(r["bm25_rank_raw"]) else int(r["bm25_rank_raw"]),
        "bm25_rank_breadcrumb": None if pd.isna(r["bm25_rank_breadcrumb"]) else int(r["bm25_rank_breadcrumb"]),
        "hit@3_raw": int(r["hit3_raw"]), "hit@3_breadcrumb": int(r["hit3_breadcrumb"]),
        "gold_coverage_raw": round(float(r["gold_coverage_raw"]), 4),
        "gold_coverage_breadcrumb": round(float(r["gold_coverage_breadcrumb"]), 4),
        "top1_chunk_raw": r["top1_raw"], "top1_chunk_breadcrumb": r["top1_breadcrumb"],
    })

summary = {
    "created_at": datetime.now(timezone.utc).isoformat(),
    "tokenizer": TIKTOKEN_ENCODING,
    "chunk_dataset": chunk_dataset_path.name,
    "n_chunks": len(chunks),
    "n_queries": len(BENCHMARK_QUERIES),
    "config": alignment["config"],
    "overall": alignment["overall"],
    "scope_split": alignment["scope_split"],
    "per_archetype": alignment["per_archetype_dict"],
    "per_query": per_query_export,
    "structural_composition": {
        "corpus_total_tokens": corpus_total,
        "corpus_pct_tokens": {c: corpus_row[f"{c}_pct"] for c in CATEGORIES},
        "corpus_blocks": {c: corpus_row[f"{c}_blocks"] for c in CATEGORIES},
        "per_document_pct_tokens": {
            r["doc_name"]: {c: r[f"{c}_pct"] for c in CATEGORIES} | {"total_tokens": int(r["total_tokens"])}
            for _, r in composition_df.iterrows()
        },
    },
    "figures": [COMPOSITION_PNG.name, GAP_PNG.name],
}

with open(SUMMARY_JSON, "w", encoding="utf-8") as f:
    json.dump(_to_builtin(summary), f, indent=2, ensure_ascii=False)

required = {"overall", "per_archetype", "per_query", "structural_composition"}
assert required <= set(json.load(open(SUMMARY_JSON, encoding="utf-8")).keys())
print(f"[SAVE] {SUMMARY_JSON} ({SUMMARY_JSON.stat().st_size / 1024:.1f} KB)")
for p in [COMPOSITION_PNG, GAP_PNG, SUMMARY_JSON]:
    print(f"   - {p} {'[OK]' if p.exists() else '[MISSING]'}")
if USE_GOOGLE_DRIVE:
    print("[INFO] Outputs were written directly to Google Drive (BASE_DIR is on Drive).")

---
# Conclusões e Próximos Passos

**O que este notebook estabelece (linha de base léxica, sem embeddings):**

1. **Composição estrutural.** As tabelas de especificação são uma fração relevante dos tokens nos guias de referência (ATC, API do Atlas 200, porting TensorFlow). Isso confirma que o chunking deve tratar tabelas como unidades atômicas e motiva metadados como `is_table` para filtragem. Os sumários (ToC) representam ruído de recuperação e podem ser filtrados.
2. **Lacuna pré-recuperação.** Perguntas com termos exatos (flags do ATC, mensagens de erro, nomes de formato) são resolvidas por busca léxica nos dois modos. A lacuna aparece nas perguntas **dependentes de escopo**, que não citam o termo do manual.
3. **Breadcrumbs** ajudam quando o termo que falta na pergunta está no **nome do documento ou da seção pai** (por exemplo, "restrições" do ATC). Quando a lacuna é de vocabulário (o usuário descreve o conceito com outras palavras), o breadcrumb não resolve.

**Como isso alimenta o roadmap do projeto:**

| Achado | Técnica de pré-consulta a avaliar |
|---|---|
| Perguntas perdidas nos dois modos (lacuna de vocabulário) | **Multi-query / RAG-Fusion** (reformulações com sinônimos) e **HyDE** |
| Perguntas genéricas demais ("quais restrições se aplicam?") | **Step-back prompting** para subir o nível de abstração e recuperar a seção de escopo |
| Perguntas de acompanhamento que dependem de contexto anterior | **RAG com memória** (reescrita da pergunta com o histórico) |
| Termos exatos (flags, códigos de erro) recuperados pelo BM25 | **Busca híbrida (BM25 + denso, RRF)** para não perder o acerto léxico ao introduzir embeddings |

As métricas exportadas em `pre_retrieval_analysis_summary.json` servem como **linha de base** para comparar, nas próximas etapas, recuperação densa, híbrida e as técnicas de pré-consulta sobre o mesmo benchmark de 25 perguntas.